# CNN Baseline vs PGSD v3 — Full Comparison

This notebook trains and evaluates a **CNN spectrogram model** against PGSD v3 across all four diagnostic tasks:

| Task | What it measures | Metric |
|------|-----------------|--------|
| T1 | Membrane tension estimation | Mean error (%) |
| T2 | Strike position recovery | Mean error (%R) |
| T3 | Skin condition index | CI MAE |
| T4 | Shell shape identification | Accuracy (%) |

## CNN Architecture
Input: 128-bin mel-spectrogram of first 500ms of signal  
4 conv layers → global average pool → task-specific head  
One model trained per task (same backbone, different head)

## PGSD v3 Results (from previous runs)
| Task | Cylinder | Hourglass | Oval | Mean |
|------|----------|-----------|------|------|
| T1 (%) | 4.41 | 3.14 | 0.32 | 2.62 |
| T2 (%R) | 0.3 | 0.3 | 0.3 | 0.3 |
| T3 (MAE) | 0.2 | 0.0 | 0.0 | 0.1 |
| T4 (%) | 100 | 100 | 100 | 100 |

## Cell 1 — Setup & Paths
**Edit `DATASET_PATH` to match your Kaggle dataset name.**

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/chenda-pgsd"   # folder containing your uploaded files
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, filtfilt
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')

PGSD_V3_PATH  = os.path.join(DATASET_PATH, 'chenda_pgsd_v3.py')
SPECTRAL_PATH = os.path.join(DATASET_PATH, 'chenda_spectral.py')

assert os.path.exists(PGSD_V3_PATH),  f'Not found: {PGSD_V3_PATH}'
assert os.path.exists(SPECTRAL_PATH), f'Not found: {SPECTRAL_PATH}'
print('Files found OK')

## Cell 2 — Load PGSD v3 (signal synthesiser + physics model)

In [ ]:
# Patch the hardcoded spectral path inside pgsd_v3 before loading
pgsd_src = open(PGSD_V3_PATH).read()
pgsd_src = pgsd_src.replace(
    'exec(open("/mnt/user-data/outputs/chenda_spectral.py")',
    f'exec(open("{SPECTRAL_PATH}")')
exec(compile(pgsd_src[:pgsd_src.rfind('\nif __name__')], 'pgsd_v3', 'exec'), globals())
print('PGSD v3 loaded OK')

# Constants
FS          = 22050
T_REF       = 3500.0
T_TENSIONS  = [3000, 3200, 3500, 3800, 4100]
R_POSITIONS = [0.00, 0.25, 0.50, 0.75, 0.90]
SHAPES_LIST = ['Cylinder', 'Hourglass', 'Oval']
SKINS = {
    'New':      ChendaDamping(0.80, 8.8e-6),
    '3 months': ChendaDamping(0.81, 1.2e-5),
    '6 months': ChendaDamping(0.83, 1.8e-5),
    '1 year':   ChendaDamping(0.86, 2.8e-5),
    'Worn':     ChendaDamping(0.92, 4.5e-5),
}
print('Constants set OK')

## Cell 3 — Mel-spectrogram feature extractor
128 mel bins, 25ms window, 10ms hop, first 500ms of signal → (1, 128, 50) tensor

In [ ]:
def hz_to_mel(hz):   return 2595.0 * np.log10(1.0 + hz / 700.0)
def mel_to_hz(mel):  return 700.0 * (10.0**(mel / 2595.0) - 1.0)

def make_melfilterbank(fs, n_fft, n_mels=128, f_min=60.0, f_max=8000.0):
    freq     = rfftfreq(n_fft, 1.0/fs)
    mel_pts  = mel_to_hz(np.linspace(hz_to_mel(f_min),
                                     hz_to_mel(min(f_max, fs/2)),
                                     n_mels + 2))
    fbank = np.zeros((n_mels, len(freq)))
    for m in range(n_mels):
        lo, c, hi = mel_pts[m], mel_pts[m+1], mel_pts[m+2]
        for k, f in enumerate(freq):
            if lo <= f <= c:   fbank[m, k] = (f - lo) / (c - lo + 1e-10)
            elif c < f <= hi:  fbank[m, k] = (hi - f) / (hi - c + 1e-10)
    return fbank   # (n_mels, n_fft//2+1)

# Pre-compute filterbank (constant across all signals)
N_FFT      = 512
HOP        = 110     # ~5ms at 22050 Hz
WIN_DUR    = 0.5     # seconds of signal to use
N_MELS     = 128
N_FRAMES   = int(np.ceil(WIN_DUR * FS / HOP))   # ~100 frames

MELBANK = make_melfilterbank(FS, N_FFT, N_MELS)
print(f'Mel filterbank: {MELBANK.shape}  →  expected input: (1, {N_MELS}, {N_FRAMES})')

def signal_to_melspec(signal, fs=FS, n_fft=N_FFT, hop=HOP,
                      win_dur=WIN_DUR, n_frames=N_FRAMES):
    """
    Convert raw signal to log mel-spectrogram.
    Returns numpy array of shape (N_MELS, N_FRAMES).
    """
    sig   = signal[:int(fs * win_dur)]
    if len(sig) < n_fft:
        sig = np.pad(sig, (0, n_fft - len(sig)))

    frames = []
    for s in range(0, int(fs * win_dur) - n_fft + 1, hop):
        frame = sig[s:s + n_fft] * np.hanning(n_fft)
        frames.append(np.abs(rfft(frame))**2)

    if not frames:
        return np.zeros((N_MELS, n_frames))

    spec    = np.array(frames).T       # (n_fft//2+1, n_frames_actual)
    mel     = MELBANK @ spec           # (N_MELS, n_frames_actual)
    log_mel = np.log(mel + 1e-10)

    # Pad or trim to fixed n_frames
    if log_mel.shape[1] < n_frames:
        log_mel = np.pad(log_mel, ((0,0),(0, n_frames - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :n_frames]

    # Normalise to zero mean, unit std
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-10)
    return log_mel    # (N_MELS, N_FRAMES)

# Quick test
test_sig = np.random.randn(FS * 2)
ms = signal_to_melspec(test_sig)
print(f'Mel-spec shape: {ms.shape}  ✓')

## Cell 4 — CNN Architecture
Follows standard MIR drum analysis CNN pattern (similar to Salamon & Bello 2017, adapted for regression/classification)

In [ ]:
class ChendaCNN(nn.Module):
    """
    4-layer CNN on mel-spectrogram for Chenda parameter estimation.

    Input  : (batch, 1, N_MELS, N_FRAMES)  →  (batch, 1, 128, ~100)
    Output : (batch, n_outputs)

    Architecture:
        Conv2d(1→32, 3×3)  + BN + ReLU + MaxPool(2×2)
        Conv2d(32→64, 3×3) + BN + ReLU + MaxPool(2×2)
        Conv2d(64→128,3×3) + BN + ReLU + MaxPool(2×2)
        Conv2d(128→128,3×3)+ BN + ReLU + GlobalAvgPool
        FC(128→64) + ReLU + Dropout(0.3)
        FC(64→n_outputs)
    """
    def __init__(self, n_outputs=1, task='regression'):
        super().__init__()
        self.task = task

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 4
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling

        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_outputs),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m = ChendaCNN(n_outputs=1)
dummy = torch.randn(4, 1, N_MELS, N_FRAMES)
out   = m(dummy)
print(f'CNN test: input {dummy.shape} → output {out.shape}')
print(f'Trainable parameters: {count_params(m):,}')

## Cell 5 — Training utility

In [ ]:
def train_cnn(X, y, task='regression', n_outputs=1,
              epochs=40, batch_size=32, lr=1e-3, verbose=True):
    """
    Train ChendaCNN on (X, y).
    X: numpy (N, 1, N_MELS, N_FRAMES)
    y: numpy (N,) for regression  or  (N,) int for classification
    Returns trained model.
    """
    Xt = torch.FloatTensor(X)
    if task == 'classification':
        yt = torch.LongTensor(y)
    else:
        yt = torch.FloatTensor(y).unsqueeze(1)

    ds     = TensorDataset(Xt, yt)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model  = ChendaCNN(n_outputs=n_outputs, task=task).to(DEVICE)
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    if task == 'classification':
        criterion = nn.CrossEntropyLoss()
    else:
        criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        sched.step()
        if verbose and (epoch+1) % 10 == 0:
            print(f'    Epoch {epoch+1:3d}/{epochs}  loss={total_loss/len(loader):.5f}')

    model.eval()
    return model


def predict_cnn(model, signal, task='regression'):
    """Run a single signal through the CNN."""
    ms  = signal_to_melspec(signal)
    x   = torch.FloatTensor(ms).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    if task == 'classification':
        return int(out.argmax(dim=1).item())
    return float(out.squeeze().item())


def build_spectrogram_dataset(task, shape=None, n=800):
    """
    Generate n synthetic signals and extract mel-spectrograms + labels.
    task: 'tension' | 'position' | 'skin' | 'shape'
    """
    np.random.seed(1234)
    dur = 1.5
    t   = np.linspace(0, dur, int(FS * dur), endpoint=False)
    X, y = [], []

    if task == 'tension':
        for _ in range(n):
            T_r  = np.random.uniform(2800, 4300)
            fr, mo, al, r, th, _ = get_spectral_basis(shape, T=T_r)
            sig, _ = synthesise_strike(fr, mo, al, r, th,
                r_s=np.random.uniform(0.1, 0.9)*R_PHYS,
                theta_s=np.random.uniform(0, 2*np.pi), t=t, snr_db=25)
            X.append(signal_to_melspec(sig)[np.newaxis])
            y.append(T_r)

    elif task == 'position':
        fr, mo, al, r, th, _ = get_spectral_basis(shape)
        for _ in range(n):
            r_r  = np.random.uniform(0.0, 0.95)
            sig, _ = synthesise_strike(fr, mo, al, r, th,
                r_s=r_r*R_PHYS,
                theta_s=np.random.uniform(0, 2*np.pi), t=t, snr_db=25)
            X.append(signal_to_melspec(sig)[np.newaxis])
            y.append(r_r)

    elif task == 'skin':
        fr, mo, _, r, th, _ = get_spectral_basis(shape)
        for eta2 in np.logspace(np.log10(8.8e-6), np.log10(4.5e-5), n):
            d   = ChendaDamping(eta1=0.80, eta2=eta2)
            al  = np.array([d.decay_rate(2*np.pi*f) for f in fr])
            sig, _ = synthesise_strike(fr, mo, al, r, th,
                r_s=0.3*R_PHYS, theta_s=0.0, t=t, snr_db=25)
            X.append(signal_to_melspec(sig)[np.newaxis])
            y.append(d.T60(2*np.pi*fr[0]))

    elif task == 'shape':
        n_each = n // 3
        for i, s in enumerate(SHAPES_LIST):
            for _ in range(n_each):
                T_r = np.random.uniform(3000, 4100)
                fr, mo, al, r, th, _ = get_spectral_basis(s, T=T_r)
                sig, _ = synthesise_strike(fr, mo, al, r, th,
                    r_s=np.random.uniform(0.1, 0.9)*R_PHYS,
                    theta_s=np.random.uniform(0, 2*np.pi), t=t, snr_db=25)
                X.append(signal_to_melspec(sig)[np.newaxis])
                y.append(i)

    return np.array(X, dtype=np.float32), np.array(y)

print('Training utilities ready')

## Cell 6 — T60 measurement helper (shared with PGSD)

In [ ]:
def measure_t60(signal, fs, f_centre, bw_frac=0.18):
    """T60 from noise-floor-gated log-envelope of bandpass signal."""
    f_nyq = fs / 2.0
    f_lo  = max(20.0, f_centre * (1 - bw_frac))
    f_hi  = min(f_nyq * 0.97, f_centre * (1 + bw_frac))
    b, a  = butter(4, [f_lo/f_nyq, f_hi/f_nyq], btype='band')
    env   = np.abs(filtfilt(b, a, signal))
    sm    = max(1, int(fs * 0.005))
    env_s = np.convolve(env, np.ones(sm)/sm, mode='same')
    t_arr = np.arange(len(signal)) / fs
    noise = env_s.max() / 31.6
    above = np.where(env_s > noise * 2)[0]
    i0 = max(int(fs*0.02), above[0])  if len(above) > 100 else int(fs*0.02)
    i1 = above[-1]                     if len(above) > 100 else int(fs*0.8)
    t_f  = t_arr[i0:i1]
    e_f  = np.log(np.maximum(env_s[i0:i1], 1e-12))
    fin  = np.isfinite(e_f)
    if fin.sum() > 20:
        alpha = max(-np.polyfit(t_f[fin], e_f[fin], 1)[0], 1e-3)
    else:
        alpha = 1.0
    return np.log(1000.0) / alpha

print('T60 helper ready')

## Cell 7 — TASK 1: Tension Estimation
CNN trained per shape on T ∈ [2800, 4300] N/m with random positions.
Evaluated on the same 5 test tensions used by PGSD.

In [ ]:
print('='*60)
print('TASK 1: TENSION ESTIMATION')
print('='*60)

cnn_t1_results = {}

for shape in SHAPES_LIST:
    print(f'\n  Shape: {shape}')
    dur = 1.5
    t   = np.linspace(0, dur, int(FS*dur), endpoint=False)
    np.random.seed(42)

    freqs_ref, modes_ref, alphas_ref, r, th, _ = get_spectral_basis(shape, T=T_REF)
    f1_ref = freqs_ref[0]

    # Build training set
    print(f'  Building training set (800 signals)...', end='', flush=True)
    t0 = time.time()
    X_tr, y_tr = build_spectrogram_dataset('tension', shape=shape, n=800)
    print(f' {time.time()-t0:.0f}s  shape={X_tr.shape}')

    # Train CNN
    print(f'  Training CNN...')
    model_t1 = train_cnn(X_tr, y_tr, task='regression', n_outputs=1, epochs=40)

    # Evaluate on test tensions
    errs = []
    print(f'  {"T_true":>7}  {"f1_true":>9}  {"T_cnn":>9}  {"err%":>7}')
    print(f'  {"─"*40}')

    for T_true in T_TENSIONS:
        freqs_t, modes_t, alphas_t, _, _, _ = get_spectral_basis(shape, T=T_true)
        sig, _ = synthesise_strike(freqs_t, modes_t, alphas_t, r, th,
                                   r_s=0.4*R_PHYS, theta_s=0.0, t=t)
        T_cnn = float(np.clip(predict_cnn(model_t1, sig), 2500, 4800))
        err   = abs(T_cnn - T_true) / T_true * 100
        errs.append(err)
        print(f'  {T_true:>7}  {freqs_t[0]:>9.2f}  {T_cnn:>9.1f}  {err:>7.2f}%')

    mean_err = float(np.mean(errs))
    print(f'  Mean error: {mean_err:.2f}%')
    cnn_t1_results[shape] = {'per_T': dict(zip(T_TENSIONS, errs)), 'mean': mean_err}

print('\n  Task 1 summary:')
for s in SHAPES_LIST:
    print(f'    CNN {s}: {cnn_t1_results[s]["mean"]:.2f}%')
print(f'    CNN Mean: {np.mean([cnn_t1_results[s]["mean"] for s in SHAPES_LIST]):.2f}%')
print(f'    PGSD Mean: 2.62%  (Oval: 0.32%)')

## Cell 8 — TASK 2: Strike Position Recovery
CNN trained on r/R ∈ [0, 0.95] with random theta.  
Key question: can learned spectrogram features recover position without mode shapes?

In [ ]:
print('='*60)
print('TASK 2: STRIKE POSITION')
print('='*60)

cnn_t2_results = {}

for shape in SHAPES_LIST:
    print(f'\n  Shape: {shape}')
    dur = 1.5
    t   = np.linspace(0, dur, int(FS*dur), endpoint=False)
    np.random.seed(77)

    freqs_ref, modes_ref, alphas_ref, r, th, _ = get_spectral_basis(shape)

    print(f'  Building training set (800 signals)...', end='', flush=True)
    t0 = time.time()
    X_tr, y_tr = build_spectrogram_dataset('position', shape=shape, n=800)
    print(f' {time.time()-t0:.0f}s')

    print(f'  Training CNN...')
    model_t2 = train_cnn(X_tr, y_tr, task='regression', n_outputs=1, epochs=40)

    errs = []
    print(f'  {"r_true/R":>10}  {"r_cnn/R":>10}  {"err%R":>8}')
    print(f'  {"─"*35}')

    for r_frac in R_POSITIONS:
        np.random.seed(77)
        sig, _ = synthesise_strike(freqs_ref, modes_ref, alphas_ref, r, th,
                                   r_s=r_frac*R_PHYS, theta_s=0.0, t=t, snr_db=25.0)
        r_cnn = float(np.clip(predict_cnn(model_t2, sig), 0.0, 0.97))
        err   = abs(r_cnn - r_frac) * 100
        errs.append(err)
        print(f'  {r_frac:>10.2f}  {r_cnn:>10.3f}  {err:>8.1f}%R')

    mean_err = float(np.mean(errs))
    print(f'  Mean error: {mean_err:.1f}%R  ({mean_err*R_PHYS*100:.2f} cm)')
    cnn_t2_results[shape] = {'per_r': dict(zip(R_POSITIONS, errs)), 'mean': mean_err}

print('\n  Task 2 summary:')
for s in SHAPES_LIST:
    print(f'    CNN {s}: {cnn_t2_results[s]["mean"]:.1f}%R')
print(f'    CNN Mean: {np.mean([cnn_t2_results[s]["mean"] for s in SHAPES_LIST]):.1f}%R')
print(f'    PGSD Mean: 0.3%R  (sub-millimetre)')

## Cell 9 — TASK 3: Skin Condition Index
CNN predicts T60 from mel-spectrogram, then CI is derived from T60.

In [ ]:
print('='*60)
print('TASK 3: SKIN CONDITION INDEX')
print('='*60)

cnn_t3_results = {}

for shape in SHAPES_LIST:
    print(f'\n  Shape: {shape}')
    dur = 2.0
    t   = np.linspace(0, dur, int(FS*dur), endpoint=False)
    np.random.seed(55)

    freqs_ref, modes_ref, _, r, th, _ = get_spectral_basis(
        shape, damping=ChendaDamping())
    f1_ref  = freqs_ref[0]
    T60_ref = ChendaDamping().T60(2*np.pi*f1_ref)

    print(f'  f1={f1_ref:.1f} Hz   T60_ref={T60_ref:.3f} s')
    print(f'  Building training set (400 signals)...', end='', flush=True)
    t0 = time.time()
    X_tr, y_tr = build_spectrogram_dataset('skin', shape=shape, n=400)
    print(f' {time.time()-t0:.0f}s')

    print(f'  Training CNN (predicts T60)...')
    model_t3 = train_cnn(X_tr, y_tr, task='regression', n_outputs=1,
                         epochs=40, lr=5e-4)

    ci_pred, ci_true_list = [], []
    print(f'  {"Skin":>12}  {"CI_true":>8}  {"CI_CNN":>8}')
    print(f'  {"─"*32}')

    for skin_name, damp in SKINS.items():
        al_s   = np.array([damp.decay_rate(2*np.pi*f) for f in freqs_ref])
        sig, _ = synthesise_strike(freqs_ref, modes_ref, al_s, r, th,
                                   r_s=0.3*R_PHYS, theta_s=0.0, t=t)
        T60_gt  = damp.T60(2*np.pi*f1_ref)
        CI_gt   = min(100, max(0, (T60_ref - T60_gt)/T60_ref*100))

        T60_cnn = float(np.clip(predict_cnn(model_t3, sig), 0.05, 5.0))
        CI_cnn  = min(100, max(0, (T60_ref - T60_cnn)/T60_ref*100))

        ci_true_list.append(CI_gt)
        ci_pred.append(CI_cnn)
        print(f'  {skin_name:>12}  {CI_gt:>8.1f}  {CI_cnn:>8.1f}')

    mae  = float(np.mean([abs(p-q) for p,q in zip(ci_pred, ci_true_list)]))
    mono = all(ci_pred[i] <= ci_pred[i+1] for i in range(len(ci_pred)-1))
    print(f'  MAE={mae:.1f}  Monotone={mono}')
    cnn_t3_results[shape] = {'mae': mae, 'monotone': mono,
                              'ci_pred': ci_pred, 'ci_true': ci_true_list}

print('\n  Task 3 summary:')
for s in SHAPES_LIST:
    print(f'    CNN {s}: MAE={cnn_t3_results[s]["mae"]:.1f}  Mono={cnn_t3_results[s]["monotone"]}')
print(f'    CNN Mean MAE: {np.mean([cnn_t3_results[s]["mae"] for s in SHAPES_LIST]):.1f}')
print(f'    PGSD Mean MAE: 0.0  (physics-correct bandpass by construction)')

## Cell 10 — TASK 4: Shell Shape Identification
CNN classifier trained across all three shapes with tension variation.  
This is the most fair test for CNN — it is a classification task it is designed for.

In [ ]:
print('='*60)
print('TASK 4: SHELL SHAPE IDENTIFICATION')
print('='*60)

# One multi-class CNN for all shapes
print('\n  Building training set (900 signals, 300 per shape)...', end='', flush=True)
t0 = time.time()
X_tr, y_tr = build_spectrogram_dataset('shape', n=900)
print(f' {time.time()-t0:.0f}s  shape={X_tr.shape}')

print('  Training CNN classifier...')
model_t4 = train_cnn(X_tr, y_tr.astype(int),
                     task='classification', n_outputs=3, epochs=40)

# Evaluate with tension variation (5 tensions × 5 trials per shape)
TRIALS = 5
dur    = 1.5
cnn_t4_results = {}

print(f'\n  Evaluating across all tensions...')
print(f'  {"True shape":>12}  {"Acc":>8}')
print(f'  {"─"*24}')

for true_shape in SHAPES_LIST:
    correct = 0
    total   = 0
    for trial in range(TRIALS):
        T_test = T_TENSIONS[trial % len(T_TENSIONS)]
        fr, mo, al, r, th, _ = get_spectral_basis(true_shape, T=T_test)
        t = np.linspace(0, dur, int(FS*dur), endpoint=False)
        np.random.seed(300 + trial)
        sig, _ = synthesise_strike(fr, mo, al, r, th,
                                   r_s=0.4*R_PHYS, theta_s=0.0, t=t)
        pred = predict_cnn(model_t4, sig, task='classification')
        if SHAPES_LIST[pred] == true_shape:
            correct += 1
        total += 1

    acc = correct/total*100
    cnn_t4_results[true_shape] = acc
    print(f'  {true_shape:>12}  {acc:>8.0f}%')

overall_acc = float(np.mean(list(cnn_t4_results.values())))
print(f'  {"Overall":>12}  {overall_acc:>8.0f}%')
print(f'\n  CNN Mean: {overall_acc:.0f}%')
print(f'  PGSD: 100%  (R² basis matching, data-free)')

## Cell 11 — Full Results Table + JSON output

In [ ]:
# ── Compute summary numbers ───────────────────────────────────────────────────
cnn_t1_mean = float(np.mean([cnn_t1_results[s]['mean'] for s in SHAPES_LIST]))
cnn_t2_mean = float(np.mean([cnn_t2_results[s]['mean'] for s in SHAPES_LIST]))
cnn_t3_mae  = float(np.mean([cnn_t3_results[s]['mae']  for s in SHAPES_LIST]))
cnn_t4_mean = float(np.mean(list(cnn_t4_results.values())))

# ── PGSD known results ────────────────────────────────────────────────────────
pgsd = {
    'T1_mean': 2.62, 'T1_oval': 0.32,
    'T2_mean': 0.3,
    'T3_mae':  0.0,
    'T4_acc':  100.0
}

# ── Print comparison table ────────────────────────────────────────────────────
print('\n' + '='*60)
print('  FULL COMPARISON: CNN vs PGSD v3')
print('='*60)
print(f'  {"":28}  {"CNN":>10}  {"PGSD":>10}')
print(f'  {"─"*52}')
print(f'  {"T1 Tension mean error (%)": <28}  {cnn_t1_mean:>10.2f}  {pgsd["T1_mean"]:>10.2f}')
print(f'  {"  (Oval only)":            <28}  {cnn_t1_results["Oval"]["mean"]:>10.2f}  {pgsd["T1_oval"]:>10.2f}')
print(f'  {"T2 Position mean error (%R)": <28}  {cnn_t2_mean:>10.1f}  {pgsd["T2_mean"]:>10.1f}')
print(f'  {"T3 Skin CI MAE":            <28}  {cnn_t3_mae:>10.1f}  {pgsd["T3_mae"]:>10.1f}')
print(f'  {"T4 Shape accuracy (%)": <28}  {cnn_t4_mean:>10.0f}  {pgsd["T4_acc"]:>10.0f}')
print(f'  {"─"*52}')
print(f'  {"Training data required": <28}  {"YES":>10}  {"NO":>10}')
print(f'  {"Physically interpretable": <28}  {"NO":>10}  {"YES":>10}')

# ── Shape-level T1 breakdown ──────────────────────────────────────────────────
print(f'\n  T1 by shape:')
for s in SHAPES_LIST:
    pgsd_by_shape = {'Cylinder': 4.41, 'Hourglass': 3.14, 'Oval': 0.32}
    print(f'    {s:>12}  CNN:{cnn_t1_results[s]["mean"]:>7.2f}%  PGSD:{pgsd_by_shape[s]:>5.2f}%')

# ── JSON block for paper update ───────────────────────────────────────────────
output = {
    'cnn_results': {
        'T1_mean_pct':  cnn_t1_mean,
        'T1_by_shape':  {s: cnn_t1_results[s]['mean'] for s in SHAPES_LIST},
        'T1_per_tension': {s: cnn_t1_results[s]['per_T'] for s in SHAPES_LIST},
        'T2_mean_pctR': cnn_t2_mean,
        'T2_by_shape':  {s: cnn_t2_results[s]['mean'] for s in SHAPES_LIST},
        'T3_mae':       cnn_t3_mae,
        'T3_by_shape':  {s: cnn_t3_results[s]['mae'] for s in SHAPES_LIST},
        'T3_monotone':  {s: cnn_t3_results[s]['monotone'] for s in SHAPES_LIST},
        'T4_acc_pct':   cnn_t4_mean,
        'T4_by_shape':  cnn_t4_results,
    },
    'pgsd_results': pgsd,
    'training_signals_per_task': {'T1': 800, 'T2': 800, 'T3': 400, 'T4': 900},
    'cnn_architecture': '4xConv2d + GAP + FC64 + FC(n_out)',
    'input_feature': f'log-mel spectrogram ({N_MELS} bins, {N_FRAMES} frames, first {WIN_DUR}s)',
}

print('\n\n' + '─'*60)
print('  JSON — copy and paste back to Claude:')
print('─'*60)
print(json.dumps(output, indent=2))
print('─'*60)

## Cell 12 — Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'font.family': 'serif'})

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('CNN Baseline vs PGSD v3 — All Four Tasks', fontsize=13, fontweight='bold')

COLORS = {'CNN': '#E24B4A', 'PGSD': '#1D9E75'}
pgsd_by_shape = {'Cylinder': 4.41, 'Hourglass': 3.14, 'Oval': 0.32}

# T1: tension per shape
ax = axes[0]
x  = np.arange(len(SHAPES_LIST))
cnn_vals  = [cnn_t1_results[s]['mean'] for s in SHAPES_LIST]
pgsd_vals = [pgsd_by_shape[s] for s in SHAPES_LIST]
ax.bar(x-0.2, cnn_vals,  0.35, label='CNN',  color=COLORS['CNN'],  alpha=0.85)
ax.bar(x+0.2, pgsd_vals, 0.35, label='PGSD', color=COLORS['PGSD'], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Cyl','Hour','Oval'])
ax.set_ylabel('Mean error (%)')
ax.set_title('T1 — Tension')
ax.legend(fontsize=9)

# T2: position per shape
ax = axes[1]
cnn_vals  = [cnn_t2_results[s]['mean'] for s in SHAPES_LIST]
pgsd_vals = [0.3, 0.3, 0.3]
ax.bar(x-0.2, cnn_vals,  0.35, color=COLORS['CNN'],  alpha=0.85, label='CNN')
ax.bar(x+0.2, pgsd_vals, 0.35, color=COLORS['PGSD'], alpha=0.85, label='PGSD')
ax.set_xticks(x); ax.set_xticklabels(['Cyl','Hour','Oval'])
ax.set_ylabel('Mean error (%R)')
ax.set_title('T2 — Position')
ax.legend(fontsize=9)

# T3: skin MAE per shape
ax = axes[2]
cnn_vals  = [cnn_t3_results[s]['mae'] for s in SHAPES_LIST]
pgsd_vals = [0.0, 0.0, 0.0]
ax.bar(x-0.2, cnn_vals,  0.35, color=COLORS['CNN'],  alpha=0.85, label='CNN')
ax.bar(x+0.2, pgsd_vals, 0.35, color=COLORS['PGSD'], alpha=0.85, label='PGSD')
ax.set_xticks(x); ax.set_xticklabels(['Cyl','Hour','Oval'])
ax.set_ylabel('CI MAE')
ax.set_title('T3 — Skin Condition')
ax.legend(fontsize=9)

# T4: shape accuracy
ax = axes[3]
methods = ['CNN', 'PGSD']
vals    = [cnn_t4_mean, 100.0]
bars = ax.bar(methods, vals, color=[COLORS['CNN'], COLORS['PGSD']], alpha=0.85, width=0.4)
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('T4 — Shape ID')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+2, f'{v:.0f}%',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/cnn_vs_pgsd.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to /kaggle/working/cnn_vs_pgsd.png')